In [ ]:
%pip -q install -U "xarray>=2024.9" "zarr>=3.1" numcodecs fsspec

import xarray as xr
from google.colab import drive
import numpy as np
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import os, subprocess, shutil

drive.mount('/content/drive')
data_dir = '/content/drive/Shared drives/team/shared/Work/Algorithm Consolidation/'


def run(cmd, check=True):
    print(f"$ {cmd}")
    rc = subprocess.call(cmd, shell=True)
    if check and rc != 0:
        raise RuntimeError(f"Command failed: {cmd} (rc={rc})")

tar_file_names = ["iceland_timeseries.tar.gz", "southern_andes_timeseries.tar.gz", "svalbard_timeseries.tar.gz"]
zarr_name = "timeseries_results.zarr"
for tar_file_name in tar_file_names:
  drive_tar = os.path.join(data_dir, tar_file_name)
  local_tar = "/content/dataset.tar.gz"
  if not os.path.exists(drive_tar):
      raise FileNotFoundError(f"Could not find tar on Drive: {drive_tar}")

  print(f"Copying archive to local SSD:\n  {drive_tar} -> {local_tar}")
  shutil.copy2(drive_tar, local_tar)
  print("Extracting locally (this is fast on the VM SSD)...")
  target_dir = os.path.join("/content", tar_file_name.strip('.tar.gz'))
  os.makedirs(target_dir, exist_ok=True)
  run(f"tar -xzf {local_tar} -C /content/{tar_file_name.strip('.tar.gz')}", check=True)

### Plot and analyse data for a chosen region

In [ ]:
region = "iceland"  # or "southern_andes" or "svalbard"

zarr_file_path = f"/content/{region}_timeseries/timeseries_results.zarr"

ds = xr.load_dataset(zarr_file_path)
ds_morris = xr.load_dataset(f"/{data_dir}/{region}_elevation_change_timeseries_morris_filtered.nc")
ds["morris_filtered"] = ds_morris.reindex(x=ds.x.values, y=ds.y.values, t=ds.t.values, method=None)["elevation_change"]
ds_morris = xr.load_dataset(f"/{data_dir}/{region}_elevation_change_timeseries_morris_unfiltered.nc")
ds["morris_unfiltered"] = ds_morris.reindex(x=ds.x.values, y=ds.y.values, t=ds.t.values, method=None)["elevation_change"]
ds

In [ ]:
import numpy as np
import plotly.graph_objects as go

def plotly_swipe(ds, var_left, var_right, t=0, vmin=-20, vmax=20,
                 colorscale='RdYlBu', n_steps=50):
    A = ds[var_left].isel(t=t)
    B = ds[var_right].isel(t=t)
    Avals = A.values
    Bvals = B.values

    # coords
    ydim, xdim = A.dims[-2], A.dims[-1]
    x = A[xdim].values if xdim in A.coords else np.arange(A.shape[-1])
    y = A[ydim].values if ydim in A.coords else np.arange(A.shape[-2])

    # Crop to non-NaN bounding box
    finite_union = np.isfinite(Avals) | np.isfinite(Bvals)
    if finite_union.any():
        rows_valid = np.where(finite_union.any(axis=1))[0]
        cols_valid = np.where(finite_union.any(axis=0))[0]
        i0, i1 = rows_valid[0], rows_valid[-1]
        j0, j1 = cols_valid[0], cols_valid[-1]

        Avals = Avals[i0:i1+1, j0:j1+1]
        Bvals = Bvals[i0:i1+1, j0:j1+1]
        y = y[i0:i1+1]
        x = x[j0:j1+1]

    # Figure size scaled to data aspect
    xspan = float(np.nanmax(x) - np.nanmin(x)) if x.size else 1.0
    yspan = float(np.nanmax(y) - np.nanmin(y)) if y.size else 1.0
    aspect_ratio = (xspan / yspan) if yspan != 0 else 1.0
    fig_height = int(700)
    fig_width  = int(np.clip(fig_height * aspect_ratio, 400, 1600))

    fig = go.Figure()

    # Left layer (full, uncropped within the trimmed extent)
    fig.add_trace(go.Heatmap(
        x=x, y=y, z=Avals, colorscale=colorscale, zmin=vmin, zmax=vmax,
        colorbar=dict(title=var_left), showscale=True, name=var_left
    ))

    # Right layer (masked to the right of the split)
    split0 = 0.5 * (float(np.nanmin(x)) + float(np.nanmax(x)))
    # mask shape must match Bvals (ny, nx); compare x per column
    mask0 = np.where(x[None, :] > split0, 1.0, np.nan)
    fig.add_trace(go.Heatmap(
        x=x, y=y, z=mask0 * Bvals, colorscale=colorscale, zmin=vmin, zmax=vmax,
        showscale=False, name=var_right
    ))

    # Guide line
    guide = dict(type="line", x0=split0, x1=split0,
                 y0=float(np.nanmin(y)), y1=float(np.nanmax(y)),
                 line=dict(dash="dash", width=1))
    fig.update_layout(shapes=[guide])

    # Slider steps (keep left layer intact)
    steps = []
    splits = np.linspace(float(np.nanmin(x)), float(np.nanmax(x)), int(n_steps))
    for s in splits:
        mask = np.where(x[None, :] > s, 1.0, np.nan)
        steps.append(dict(
            method="update",
            args=[
                {"z": [Avals, mask * Bvals]},
                {"shapes": [dict(type="line", x0=s, x1=s,
                                 y0=float(np.nanmin(y)), y1=float(np.nanmax(y)),
                                 line=dict(dash="dash", width=1))]}
            ],
            label=f"{s:.2f}"
        ))

    fig.update_layout(
        title=f"Swipe compare: {var_left} | {var_right}  (t={ds.t.values[t]})",
        xaxis=dict(scaleanchor="y", scaleratio=1),   # equal units
        yaxis=dict(constrain="domain"),
        sliders=[dict(
            active=len(steps)//2,
            currentvalue={"prefix": "Swipe x: "},
            pad={"t": 10},
            steps=steps
        )],
        margin=dict(l=0, r=0, t=40, b=0),
        height=fig_height,
        width=fig_width
    )

    # Optional: lock visible ranges tightly to data (prevents extra padding)
    fig.update_xaxes(range=[float(np.nanmin(x)), float(np.nanmax(x))])
    fig.update_yaxes(range=[float(np.nanmin(y)), float(np.nanmax(y))])

    fig.show()

In [ ]:
plotly_swipe(ds, "medians", "radial_basis_function", t=20, vmin=-20, vmax=20)

In [ ]:
plotly_swipe(ds, "medians", "weighted_median", t=20, vmin=-20, vmax=20)

In [ ]:
plotly_swipe(ds, "medians", "morris_unfiltered", t=20, vmin=-20, vmax=20)

In [ ]:
def plot_means_per_timestep(ds, start_at_zero=False):
    # pick variables that have time dim 't'
    vars_with_t = [v for v in ds.data_vars if v != 'spatial_ref' and 't' in ds[v].dims]
    if not vars_with_t:
        raise ValueError("No data variables with a 't' dimension were found.")

    # Build figure
    fig = go.Figure()

    for var in vars_with_t:
        da = ds[var]

        # mean over ALL non-time dims (robust: works for 2D/3D+ arrays); skip NaNs
        mean_ts = da.mean(dim=[d for d in da.dims if d != 't'], skipna=True)

        # if dask-backed, compute now
        if hasattr(mean_ts.data, "compute"):
            mean_ts = mean_ts.copy(data=mean_ts.data.compute())

        tvals = mean_ts['t'].values
        yvals = np.asarray(mean_ts.values, dtype=float)

        fig.add_trace(go.Scatter(
            x=tvals, y=yvals - yvals[0] if start_at_zero else yvals, mode="lines+markers", name=var
        ))

    fig.update_layout(
        title="Spatial mean per timestep",
        xaxis_title="t",
        yaxis_title="Mean value",
        template="plotly_white",
        hovermode="x unified",
        legend_title_text="Variable",
        margin=dict(l=40, r=10, t=50, b=40),
    )
    # If your 't' coordinate is datetime-like:
    fig.update_xaxes(type="date" if np.issubdtype(ds['t'].dtype, np.datetime64) else None)

    fig.show()

plot_means_per_timestep(ds)
plot_means_per_timestep(ds, start_at_zero=True)